# 🔧 СБОРКА APK С ПОДРОБНОЙ ДИАГНОСТИКОЙ

Эта версия показывает точные ошибки и проблемы при сборке

In [ ]:
# ЯЧЕЙКА: ДИАГНОСТИКА И СБОРКА APK
# С подробным логированием и проверками

import os
import subprocess
import sys
import glob
from pathlib import Path
from datetime import datetime

# Цвета для вывода
class Colors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

def print_section(title):
    print(f"\n{Colors.HEADER}{'='*60}{Colors.ENDC}")
    print(f"{Colors.HEADER}{title:^60}{Colors.ENDC}")
    print(f"{Colors.HEADER}{'='*60}{Colors.ENDC}\n")

def print_ok(msg):
    print(f"{Colors.OKGREEN}✓{Colors.ENDC} {msg}")

def print_error(msg):
    print(f"{Colors.FAIL}✗{Colors.ENDC} {msg}")

def print_warning(msg):
    print(f"{Colors.WARNING}⚠{Colors.ENDC} {msg}")

def print_info(msg):
    print(f"{Colors.OKCYAN}ℹ{Colors.ENDC} {msg}")

# ============================================
# 1. ПРОВЕРКА ОКРУЖЕНИЯ
# ============================================

print_section("ДИАГНОСТИКА ОКРУЖЕНИЯ")

print(f"Python версия: {sys.version}")
print(f"Текущая директория: {os.getcwd()}")
print(f"Дата/время: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

# Проверяем место на диске
import shutil
total, used, free = shutil.disk_usage("/")
free_gb = free / (1024**3)
print(f"Свободное место на диске: {free_gb:.1f} GB")

if free_gb < 5:
    print_error(f"Недостаточно места! Требуется минимум 5 GB")
    print("Очистите место на диске и повторите попытку")
    sys.exit(1)
else:
    print_ok(f"Достаточно места на диске")

print()

# ============================================
# 2. ПОИСК ПРОЕКТА
# ============================================

print_section("ПОИСК ПРОЕКТА")

project_dir = None
possible_paths = [
    '/content/opencode',
    '/content/magic-artifact',
    '/workspace',
]

print("Поиск в стандартных местах:")
for path in possible_paths:
    if os.path.exists(path):
        buildozer_path = os.path.join(path, 'buildozer.spec')
        if os.path.exists(buildozer_path):
            print_ok(f"{path}")
            project_dir = path
        else:
            print_warning(f"{path} (но нет buildozer.spec)")
    else:
        print(f"  ✗ {path} (не существует)")

# Дополнительный поиск
if not project_dir:
    print("\nРасширенный поиск...")
    for item in os.listdir('/content'):
        path = f'/content/{item}'
        if os.path.isdir(path) and os.path.exists(f'{path}/buildozer.spec'):
            print_ok(f"Найден: {path}")
            project_dir = path
            break

if not project_dir:
    print_error("buildozer.spec не найден!")
    print("\nДоступные папки в /content:")
    for item in os.listdir('/content'):
        print(f"  • {item}")
    sys.exit(1)

print_ok(f"Проект найден: {project_dir}")
os.chdir(project_dir)
print_ok(f"Переходим в: {os.getcwd()}")

print()

# ============================================
# 3. ПРОВЕРКА СТРУКТУРЫ
# ============================================

print_section("ПРОВЕРКА СТРУКТУРЫ ПРОЕКТА")

required_files = [
    'main.py',
    'buildozer.spec',
    'src/spell_manager.py',
    'src/voice_manager.py',
    'src/media_player.py',
]

for file in required_files:
    if os.path.exists(file):
        size = os.path.getsize(file)
        print_ok(f"{file} ({size} bytes)")
    else:
        print_error(f"{file} НЕ НАЙДЕН!")

print()

# ============================================
# 4. ПРОВЕРКА ЗАВИСИМОСТЕЙ
# ============================================

print_section("ПРОВЕРКА ЗАВИСИМОСТЕЙ")

dependencies = [
    ('buildozer', 'buildozer --version'),
    ('java', 'java -version'),
    ('python', 'python3 --version'),
]

for name, cmd in dependencies:
    try:
        result = subprocess.run(cmd.split(), capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            output = (result.stdout + result.stderr).strip().split('\n')[0]
            print_ok(f"{name}: {output}")
        else:
            print_error(f"{name}: {result.stderr}")
    except Exception as e:
        print_error(f"{name}: {e}")

print()

# ============================================
# 5. ПРОВЕРКА ПЕРЕМЕННЫХ ОКРУЖЕНИЯ
# ============================================

print_section("ПЕРЕМЕННЫЕ ОКРУЖЕНИЯ")

android_vars = {
    'ANDROID_SDK_ROOT': os.path.expanduser("~/android-sdk"),
    'ANDROID_NDK_ROOT': os.path.expanduser("~/android-sdk/ndk/25.1.8937393"),
}

for var_name, var_value in android_vars.items():
    if os.environ.get(var_name):
        print_ok(f"{var_name} установлена")
        print_info(f"  Значение: {os.environ.get(var_name)}")
    else:
        print_warning(f"{var_name} не установлена")
        print_info(f"  Устанавливаю: {var_value}")
        os.environ[var_name] = var_value

print_ok("Переменные окружения установлены")
print()

# ============================================
# 6. ЧИСТКА ПЕРЕД СБОРКОЙ
# ============================================

print_section("ПОДГОТОВКА К СБОРКЕ")

if os.path.exists('bin'):
    print_warning("Папка bin существует, очищаю...")
    import shutil
    try:
        shutil.rmtree('bin')
        print_ok("Папка bin удалена")
    except:
        print_warning("Не удалось удалить bin, сборка перезапишет файлы")

if os.path.exists('.buildozer'):
    print_warning("Папка .buildozer существует")
    print_info("Оставляю для ускорения сборки")

print_ok("Подготовка завершена")
print()

# ============================================
# 7. САМА СБОРКА
# ============================================

print_section("🔨 СБОРКА APK")

print(f"Запуск: buildozer android debug\n")

try:
    # Запускаем buildozer с полным выводом
    process = subprocess.Popen(
        ['buildozer', 'android', 'debug'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    
    # Читаем вывод построчно
    output_lines = []
    error_found = False
    
    for line in process.stdout:
        # Выводим каждую строку
        print(line.rstrip())
        output_lines.append(line)
        
        # Отслеживаем ошибки
        if 'error' in line.lower() or 'failed' in line.lower():
            error_found = True
    
    # Ждём завершения
    return_code = process.wait()
    
    print()
    print(f"{'-'*60}")
    print()
    
    # ============================================
    # 8. АНАЛИЗ РЕЗУЛЬТАТА
    # ============================================
    
    print_section("АНАЛИЗ РЕЗУЛЬТАТА")
    
    if return_code == 0:
        print_ok("buildozer завершилась успешно")
        
        # Ищем APK
        apk_files = glob.glob('bin/*.apk')
        if apk_files:
            for apk in apk_files:
                size_mb = os.path.getsize(apk) / (1024*1024)
                print_ok(f"APK найден: {os.path.basename(apk)}")
                print_info(f"  Размер: {size_mb:.1f} MB")
                print_info(f"  Путь: {apk}")
            print()
            print_ok("✅ СБОРКА УСПЕШНА!")
        else:
            print_error("APK файл не найден!")
            print_warning("buildozer завершилась с кодом 0, но APK не создан")
            
    else:
        print_error(f"buildozer завершилась с ошибкой (код: {return_code})")
        print()
        
        # Анализ ошибок
        output_text = ''.join(output_lines)
        
        print(f"{Colors.FAIL}ОШИБКИ И ПРЕДУПРЕЖДЕНИЯ:{Colors.ENDC}")
        print()
        
        error_lines = []
        for line in output_lines:
            if any(keyword in line.lower() for keyword in ['error', 'failed', 'fatal', 'exception', 'traceback']):
                error_lines.append(line.strip())
        
        if error_lines:
            for i, line in enumerate(error_lines[-20:], 1):  # Последние 20 ошибок
                print(f"{Colors.FAIL}{i}. {line}{Colors.ENDC}")
        else:
            print("Ошибки не найдены в выводе")
        
        print()
        print(f"{Colors.WARNING}РЕКОМЕНДАЦИИ:{Colors.ENDC}")
        
        if 'out of memory' in output_text.lower():
            print_warning("Недостаточно памяти - перезагрузи Colab и повтори")
        elif 'timeout' in output_text.lower():
            print_warning("Timeout - медленное соединение, повтори попытку")
        elif 'gradle' in output_text.lower() or 'gradle' in output_text.lower():
            print_warning("Ошибка Gradle - попробуй перезагрузить окружение")
        elif 'sdk' in output_text.lower():
            print_warning("Проблема с Android SDK - убедись что ячейка 3 прошла успешно")
        else:
            print_warning("Неизвестная ошибка - смотри вывод выше")
        
        print()
        print(f"{Colors.WARNING}ДЕЙСТВИЯ:{Colors.ENDC}")
        print("1. Внимательно прочитай ошибки выше")
        print("2. Нажми Runtime → Restart runtime")
        print("3. Повтори все ячейки по порядку")
        print("4. Если ошибка повторится, используй Docker или WSL2")
        
except KeyboardInterrupt:
    print()
    print_error("Сборка прервана пользователем")
except Exception as e:
    print()
    print_error(f"Неожиданная ошибка: {e}")
    import traceback
    traceback.print_exc()

print()
print(f"{'-'*60}")
print(f"Конец диагностики: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'-'*60}")